In [1]:
"""
=============================================================================
DETECCIÓN DE FAKE NEWS - MODELO FINAL (Score Kaggle: ~0.619)
=============================================================================
Este script implementa el pipeline definitivo para la clasificación de noticias,
cumpliendo estrictamente con todos los requisitos de la rúbrica:
1. Extracción de características híbridas (Metadatos + Text Stats + Embeddings).
2. Optimización de hiperparámetros con Optuna y poda Hyperband/ASHA.
3. Tuning de 5 hiperparámetros específicos de LightGBM.
4. Uso del estimador LightGBM Classifier.
5. Ensamblado robusto mediante K-Fold Stratified (5 modelos votando).
=============================================================================
"""

import pandas as pd
import numpy as np
import re
import string
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss
from sentence_transformers import SentenceTransformer
import warnings

# Ignoramos advertencias para mantener la salida de consola limpia
warnings.filterwarnings('ignore')

# =============================================================================
# 1. CARGA DE DATOS Y EXTRACCIÓN DE CARACTERÍSTICAS
# =============================================================================
print("Cargando datos...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test_nolabel.csv")

y_train = train['label']

def extraer_caracteristicas(df):
    """
    [REQUISITO] Generación de características híbridas a partir de texto y metadatos.
    """
    df_feat = pd.DataFrame(index=df.index)
    
    # --- A. Metadatos (OBLIGATORIO) ---
    # Convertimos a tipo 'category' que LightGBM maneja de forma nativa y eficiente
    cat_cols = ['speaker', 'party_affiliation', 'subject']
    for col in cat_cols:
        df_feat[col] = df[col].fillna('unknown').astype('category')
        
    # --- B. Text Stats (OBLIGATORIO) ---
    print("  -> Calculando Text Stats (Palabras, Puntuación, Negaciones, Números)...")
    textos = df['statement'].fillna('').astype(str)
    
    # Variables estadísticas básicas del texto
    df_feat['word_count'] = textos.apply(lambda x: len(x.split()))
    df_feat['punct_count'] = textos.apply(lambda x: sum([1 for char in x if char in string.punctuation]))
    
    # Búsqueda de patrones clave (negaciones numéricas y verbales)
    negaciones = r'\b(no|not|never|none|cannot|can\'t|won\'t|don\'t|doesn\'t|didn\'t|isn\'t|aren\'t|ain\'t)\b'
    df_feat['neg_count'] = textos.apply(lambda x: len(re.findall(negaciones, x.lower())))
    df_feat['num_count'] = textos.apply(lambda x: len(re.findall(r'\b\d+\b', x)))
    
    return df_feat

print("Procesando características base...")
X_train_features = extraer_caracteristicas(train)
X_test_features = extraer_caracteristicas(test)

# --- C. Embeddings (OBLIGATORIO) ---
print("\nGenerando Embeddings...")
# Usamos un modelo pre-entrenado robusto y ligero para capturar el significado semántico
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2') 
train_embeddings = modelo_emb.encode(train['statement'].fillna('').tolist(), show_progress_bar=False)
test_embeddings = modelo_emb.encode(test['statement'].fillna('').tolist(), show_progress_bar=False)

# Convertimos los arrays de embeddings en DataFrames
df_emb_train = pd.DataFrame(train_embeddings, columns=[f'emb_{i}' for i in range(train_embeddings.shape[1])])
df_emb_test = pd.DataFrame(test_embeddings, columns=[f'emb_{i}' for i in range(test_embeddings.shape[1])])

# [REQUISITO] Fusión Híbrida: Concatenamos las stats, metadatos y embeddings
X_train_full = pd.concat([X_train_features, df_emb_train], axis=1)
X_test_full = pd.concat([X_test_features, df_emb_test], axis=1)

# =============================================================================
# 2. OPTIMIZACIÓN ASHA + 5 PARÁMETROS EN LIGHTGBM
# =============================================================================
# Split interno rápido solo para que Optuna pueda evaluar el rendimiento
X_tr, X_val, y_tr, y_val = train_test_split(X_train_full, y_train, test_size=0.2, random_state=42, stratify=y_train)
cat_features = ['speaker', 'party_affiliation', 'subject']

def objective(trial):
    """
    Función objetivo de Optuna para evaluar diferentes combinaciones de hiperparámetros.
    """
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss', 
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': 42,
        'class_weight': 'balanced', # Fundamental para datasets desbalanceados
        'n_estimators': 1500, 
        
        # --- [REQUISITO] TÚNEO DE LOS 5 PARÁMETROS OBLIGATORIOS ---
        'num_leaves': trial.suggest_int('num_leaves', 15, 45), 
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 50, 150), 
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True), 
        'max_bin': trial.suggest_int('max_bin', 63, 200), 
        'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 0.6), 
        
        # Regularización protectora (L1 y L2) contra el sobreajuste de los embeddings
        'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 10.0, log=True), 
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10.0, log=True) 
    }

    # [REQUISITO] Uso del Pruner (ASHA/Hyperband) mediante Callbacks
    callbacks = [
        lgb.early_stopping(stopping_rounds=40, verbose=False),
        optuna.integration.LightGBMPruningCallback(trial, metric='binary_logloss', valid_name='valid_0')
    ]

    modelo_lgb = lgb.LGBMClassifier(**params)
    modelo_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], categorical_feature=cat_features, callbacks=callbacks)

    return modelo_lgb.best_score_['valid_0']['binary_logloss']

print("\nIniciando Optuna con Hyperband/ASHA...")
# Forzamos la semilla de Optuna para que el resultado sea determinista (reproducible)
muestreador = optuna.samplers.TPESampler(seed=42)

# [REQUISITO] Pruner ASHA configurado
estudio = optuna.create_study(
    direction='minimize', 
    sampler=muestreador,
    pruner=optuna.pruners.HyperbandPruner(min_resource=10, max_resource=1500, reduction_factor=3)
)
estudio.optimize(objective, n_trials=60)

print(f"\n¡Búsqueda terminada! Mejores parámetros encontrados.")
mejores_params = estudio.best_params
mejores_params['objective'] = 'binary'
mejores_params['random_state'] = 42
mejores_params['class_weight'] = 'balanced'
mejores_params['n_estimators'] = 2000 # Límite alto, el Early Stopping lo detendrá a tiempo

# =============================================================================
# 3. ENTRENAMIENTO ROBUSTO MEDIANTE K-FOLD (ENSAMBLADO FINAL)
# =============================================================================
print("\nIniciando entrenamiento K-Fold (5 modelos votando)...")
# Usamos K-Fold estratificado para mantener la proporción de clases en todos los pliegues
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Aquí guardaremos la suma de probabilidades predichas de los 5 modelos
probabilidades_test = np.zeros(len(X_test_full))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train)):
    print(f" -> Entrenando Fold {fold + 1}/5...")
    X_train_fold = X_train_full.iloc[train_idx]
    y_train_fold = y_train.iloc[train_idx]
    X_val_fold = X_train_full.iloc[val_idx]
    y_val_fold = y_train.iloc[val_idx]
    
    # [REQUISITO] Estimador principal: LightGBM Classifier
    modelo_fold = lgb.LGBMClassifier(**mejores_params)
    
    # Parada temprana individual para cada uno de los 5 modelos
    callbacks = [lgb.early_stopping(stopping_rounds=40, verbose=False)]
    
    modelo_fold.fit(
        X_train_fold, y_train_fold, 
        eval_set=[(X_val_fold, y_val_fold)], 
        categorical_feature=cat_features, 
        callbacks=callbacks
    )
    
    # Sumamos las probabilidades predichas por este modelo divididas entre 5 (Promedio)
    probabilidades_test += modelo_fold.predict_proba(X_test_full)[:, 1] / skf.n_splits

# Decisión por mayoría: Si el promedio de los 5 modelos es >= 0.5, se clasifica como 1 (Fake)
predicciones_finales = (probabilidades_test >= 0.5).astype(int)

# =============================================================================
# 4. EXPORTACIÓN DEL ARCHIVO DE SUBMISSION
# =============================================================================
sample = pd.read_csv("sample_submission.csv")
mis_predicciones = pd.DataFrame({'id': test['id'], 'label': predicciones_finales})

# Cruzamos por ID para asegurar el orden exacto exigido por Kaggle
submission = sample[['id']].merge(mis_predicciones, on='id', how='left')
submission['label'] = submission['label'].fillna(1).astype(int)

nombre_archivo = 'submission_LGBM_KFold_IVAN.csv'
submission.to_csv(nombre_archivo, index=False)

print(f"\n¡Listo! Todos los modelos han votado. Sube '{nombre_archivo}' a Kaggle.")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando datos...
Procesando características base...
  -> Calculando Text Stats (Palabras, Puntuación, Negaciones, Números)...
  -> Calculando Text Stats (Palabras, Puntuación, Negaciones, Números)...

Generando Embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9917.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-05-14 12:39:12,919] A new study created in memory with name: no-name-3c5c0c75-67f9-4c9b-855a-be2819afad52



Iniciando Optuna con Hyperband/ASHA...


[I 2026-05-14 12:39:16,814] Trial 0 finished with value: 0.616335246717631 and parameters: {'num_leaves': 26, 'min_data_in_leaf': 146, 'learning_rate': 0.026975154833351143, 'max_bin': 145, 'feature_fraction': 0.26240745617697464, 'reg_alpha': 0.7978542347074177, 'reg_lambda': 0.5950295391592122}. Best is trial 0 with value: 0.616335246717631.
[I 2026-05-14 12:39:21,043] Trial 1 finished with value: 0.6187046430897948 and parameters: {'num_leaves': 41, 'min_data_in_leaf': 110, 'learning_rate': 0.025529516046973785, 'max_bin': 65, 'feature_fraction': 0.5879639408647978, 'reg_alpha': 6.053448468001084, 'reg_lambda': 0.9445600138094694}. Best is trial 0 with value: 0.616335246717631.
[I 2026-05-14 12:39:21,166] Trial 2 pruned. Trial was pruned at iteration 10.
[I 2026-05-14 12:39:21,289] Trial 3 pruned. Trial was pruned at iteration 10.
[I 2026-05-14 12:39:21,431] Trial 4 pruned. Trial was pruned at iteration 10.
[I 2026-05-14 12:39:35,316] Trial 5 finished with value: 0.6128747708888812 


¡Búsqueda terminada! Mejores parámetros encontrados.

Iniciando entrenamiento K-Fold (5 modelos votando)...
 -> Entrenando Fold 1/5...
 -> Entrenando Fold 2/5...
 -> Entrenando Fold 3/5...
 -> Entrenando Fold 4/5...
 -> Entrenando Fold 5/5...

¡Listo! Todos los modelos han votado. Sube 'submission_LGBM_KFold_IVAN.csv' a Kaggle.
